In [1]:
# !pip install folium httpx

In [2]:
# !pip install vesselapi

In [11]:

import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("VESSELAPI_KEY")
# print(API_KEY)


In [16]:


from vessel_api_python import VesselClient

client = VesselClient(api_key=API_KEY)
position = client.vessels.position(
    vessel_id="352001336"
)

# Async usage
from vesselapi import AsyncVesselClient

async_client = AsyncVesselClient(api_key=API_KEY)
position = await async_client.vessels.position(253041000, "mmsi")

ImportError: cannot import name 'AsyncVesselClient' from 'vesselapi' (/Users/mkl/Documents/seantinel/.venv/lib/python3.13/site-packages/vesselapi/__init__.py)

In [ ]:
import httpx
import bz2
import json
import folium
from datetime import datetime

# Configurație fixă conform datelor tale
API_KEY = "9175f417e278a561fd79e6f0439dd2e06d4950a1f18a5c76314f0383935231aa"
URL = f"https://data.aishub.net/ws.php?&format=1&output=json&compress=3&latmin=42.0&latmax=47.5&lonmin=27.0&lonmax=41.0"

async def fetch_ais_data():
    async with httpx.AsyncClient() as client:
        try:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Se descarcă datele de la AISHub...")
            # Trimitem un User-Agent pentru a evita blocajele de securitate
            response = await client.get(URL, timeout=20.0, headers={"User-Agent": "AIS-Viewer-Colab"})

            if response.status_code == 200:
                # AISHub trimite BZIP2 când compress=3
                raw_data = bz2.decompress(response.content)
                data = json.loads(raw_data)

                # Verificăm dacă avem date valide (AISHub pune navele în al doilea element al listei)
                if isinstance(data, list) and len(data) > 1:
                    return data[1]
                else:
                    print("⚠️ Notă: API-ul a răspuns, dar nu sunt nave în zona selectată sau cheia este limitată.")
                    if isinstance(data, list) and len(data) > 0:
                        print(f"Status API: {data[0]}")
            else:
                print(f"❌ Eroare HTTP: {response.status_code}")
        except Exception as e:
            print(f"❌ Eroare la procesare: {e}")
    return []

async def draw_map():
    vessels = await fetch_ais_data()

    # Centrare pe Marea Neagră / România
    m = folium.Map(
        location=[44.17, 30.0],
        zoom_start=7,
        tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
        attr='&copy; CartoDB'
    )

    for v in vessels:
        lat, lon = v.get("LATITUDE"), v.get("LONGITUDE")
        if lat and lon:
            name = v.get("NAME", "Unknown").strip()
            mmsi = v.get("MMSI")
            speed = v.get("SPEED", 0)
            course = v.get("COURSE", 0)

            # Popup personalizat
            html = f"""
            <div style="font-family: Arial; font-size: 12px; width: 160px;">
                <h4 style="margin:0 0 5px 0; color: #2c3e50;">{name}</h4>
                <table style="width:100%">
                    <tr><td><b>MMSI:</b></td><td>{mmsi}</td></tr>
                    <tr><td><b>Viteză:</b></td><td>{speed} kn</td></tr>
                    <tr><td><b>Curs:</b></td><td>{course}°</td></tr>
                </table>
            </div>
            """

            folium.CircleMarker(
                location=[lat, lon],
                radius=5,
                color='blue' if speed > 0.5 else 'red', # Albastru dacă se mișcă, roșu dacă stă
                fill=True,
                fill_opacity=0.7,
                popup=folium.Popup(html, max_width=200),
                tooltip=name
            ).add_to(m)

    print(f"✅ Harta a fost generată cu {len(vessels)} nave.")
    return m

# Rulare în celulă
map_result = await draw_map()
map_result

[13:00:39] Se descarcă datele de la AISHub...
❌ Eroare la procesare: Expecting value: line 1 column 1 (char 0)
✅ Harta a fost generată cu 0 nave.


In [ ]:
import httpx
import bz2
import json

# CONFIGURARE (Ajustează aici!)
# Înlocuiește 'USER_TAU' cu numele tău de cont de pe site-ul unde ai făcut poza
USERNAME = "ca8ff8160b1bfe189464b6bd2d151ee66fabfe0f8d47329c10c3eb19f3ddbb9b"
# Dacă 'ca8ff...' este API KEY, asigură-te că URL-ul o primește unde trebuie.

URL = f"https://data.aishub.net/ws.php?username={USERNAME}&format=1&output=json&compress=3&latmin=42.0&latmax=47.5&lonmin=27.0&lonmax=41.0"

async def test_connection():
    async with httpx.AsyncClient() as client:
        print("🔍 Se verifică conexiunea...")
        try:
            response = await client.get(URL, timeout=15.0)
            print(f"📡 Status Code: {response.status_code}")

            # Dacă e 200, încercăm să citim conținutul
            if response.status_code == 200:
                try:
                    # Încercăm să vedem dacă e text simplu (eroare) sau binar (date bzip2)
                    if response.content.startswith(b'{') or response.content.startswith(b'['):
                        # E JSON necomprimat
                        data = json.loads(response.content)
                        print("📝 Serverul a trimis JSON direct (fără compresie):")
                        print(data)
                    else:
                        # E probabil BZIP2
                        raw = bz2.decompress(response.content)
                        data = json.loads(raw)
                        print(f"✅ Succes! Am primit date pentru {len(data[1])} nave.")
                except Exception as e:
                    print(f"⚠️ Serverul a trimis ceva ce nu pot dezarhiva: {response.content[:100]}")
                    print(f"Eroare detaliată: {e}")
            else:
                print(f"❌ Serverul a respins cererea. Mesaj: {response.text}")

        except Exception as e:
            print(f"❌ Nu am putut contacta serverul: {e}")

await test_connection()

🔍 Se verifică conexiunea...
📡 Status Code: 200
⚠️ Serverul a trimis ceva ce nu pot dezarhiva: b'BZh51AY&SY]\xc1\x9f\x8e\x00\x00/\x9f\x80p\x04\x7f\xf0#\xe3\x9e\n\xbf\'\xdf\x8a \x00\x92\x15\x00hh\xc4\x00i\xa0\x00\x1a2\rT\xfc\x84\x9bS\xc8\x80\xcdO(\xda\x8d=\'\xa8\x00\xd1/\x18\x07\xca6\x8b\xb1hp"mA\xbb4\x1f:\xdc\x8a\xc5\xb6)\xd3\xa5\xe9c\x03\xd8\xc4\xa2\x95\x14\xea\xa2\n\xce\x07-'
Eroare detaliată: list index out of range


In [ ]:
import httpx
import bz2
import json

# Datele tale
API_KEY = "9175f417e278a561fd79e6f0439dd2e06d4950a1f18a5c76314f0383935231aa"
URL = f"https://data.aishub.net/ws.php?username={API_KEY}&format=1&output=json&compress=3&latmin=42.0&latmax=47.5&lonmin=27.0&lonmax=41.0"

async def debug_ais_structure():
    async with httpx.AsyncClient() as client:
        try:
            print("🚀 Descarc datele...")
            response = await client.get(URL, timeout=15.0)

            if response.status_code == 200:
                # Decomprimăm datele primite
                raw_json = bz2.decompress(response.content)
                data = json.loads(raw_json)

                print("-" * 30)
                print("📊 STRUCTURA DATELOR PRIMITE:")
                print(f"Tipul datelor: {type(data)}")

                # Afișăm primele elemente pentru a înțelege structura
                if isinstance(data, list):
                    print(f"Număr elemente în listă: {len(data)}")
                    for i, item in enumerate(data):
                        print(f"\nElement [{i}]:")
                        # Printăm doar o parte din element ca să nu umplem ecranul
                        print(str(item)[:300] + "...")
                else:
                    print("Datele nu sunt o listă. Conținut:")
                    print(str(data)[:500])
                print("-" * 30)

            else:
                print(f"Eroare HTTP: {response.status_code}")
        except Exception as e:
            print(f"Eroare: {e}")

await debug_ais_structure()


🚀 Descarc datele...
------------------------------
📊 STRUCTURA DATELOR PRIMITE:
Tipul datelor: <class 'list'>
Număr elemente în listă: 1

Element [0]:
{'ERROR': True, 'USERNAME': '9175f417e278a561fd79e6f0439dd2e06d4950a1f18a5c76314f0383935231aa', 'FORMAT': 'HUMAN', 'ERROR_MESSAGE': 'Invalid username or password!'}...
------------------------------
